In [3]:
import os
import zipfile

zip_filename = 'plantvillage.zip'
target_dir = './tomato_dataset'

# will Download only if zip doesn't exist
if not os.path.exists(zip_filename):
    print("Downloading dataset...")
    !kaggle datasets download -d mohitsingh1804/plantvillage
    print("Download complete!")
else:
    print(f"'{zip_filename}' already exists")

#Extract  only if it hasn't been extracted 
if not os.path.exists(target_dir):
    print("Extracting dataset...")
    os.makedirs(target_dir, exist_ok=True)
    with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
        zip_ref.extractall(target_dir)
    print("Extracted")
else:
    print("already extracted")

'plantvillage.zip' already exists
already extracted


In [16]:
import torch
import torch.nn as nn
from torchvision.models import googlenet, GoogLeNet_Weights
import torchvision
from torchvision.datasets import ImageFolder
import torchvision.transforms.v2 as T
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [5]:
dataset_path = "./tomato_dataset/PlantVillage"

train_dir = f"./tomato_dataset/PlantVillage/train"

val_dir = f"./tomato_dataset/PlantVillage/val"

In [10]:
# Compute Mean and Std 

mean = torch.tensor([0.4496, 0.4651, 0.4003])
std = torch.tensor([0.1656, 0.1448, 0.1831])
dataset_path = "./tomato_dataset/PlantVillage"

train_dir = f"{dataset_path}/train"
val_dir = f"{dataset_path}/val"


# Training Transform

train_transform = T.Compose([

    # Data Augmentation
    T.RandomResizedCrop(
        size=(224, 224),
        scale=(0.8, 1.0),
        ratio=(0.9, 1.1)
    ),

    T.RandomHorizontalFlip(p=0.5),

    T.RandomRotation(degrees=15),

    T.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05
    ),

    # Convert to Tensor
    T.ToImage(),

    T.ToDtype(torch.float32, scale=True),

    # Normalize
    T.Normalize(
        mean=mean.tolist(),
        std=std.tolist()
    )
])


# Validation Transform

val_transform = T.Compose([

    T.Resize((224, 224)),

    T.ToImage(),

    T.ToDtype(torch.float32, scale=True),

    T.Normalize(
        mean=mean.tolist(),
        std=std.tolist()
    )
])


# Test-Time Augmentation 


tta_transforms = [

    T.Compose([
        T.Resize((224,224)),
        T.ToImage(),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(mean.tolist(), std.tolist())
    ]),

    T.Compose([
        T.Resize((224,224)),
        T.RandomHorizontalFlip(p=1.0),
        T.ToImage(),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(mean.tolist(), std.tolist())
    ]),

    T.Compose([
        T.Resize((224,224)),
        T.RandomRotation(10),
        T.ToImage(),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(mean.tolist(), std.tolist())
    ])
]

In [11]:
# Dataset


train_data = ImageFolder(
    train_dir,
    transform=train_transform
)

val_data = ImageFolder(
    val_dir,
    transform=val_transform
)

In [12]:

# DataLoader


train_loader = DataLoader(
    train_data,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_data,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

print("Classes:", train_data.classes)
print("Training Images:", len(train_data))
print("Validation Images:", len(val_data))

Classes: ['Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Target_Spot', 'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___Tomato_mosaic_virus', 'Tomato___healthy']
Training Images: 14529
Validation Images: 3631


In [13]:
device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

print(device)

mps


In [14]:
class TomatoGoogLeNet(nn.Module):

    def __init__(self, num_classes=10):
        super().__init__()

        backbone = googlenet(
            weights=GoogLeNet_Weights.DEFAULT,
            aux_logits=True
        )

        backbone.fc = nn.Linear(
            backbone.fc.in_features,
            num_classes
        )

        self.model = backbone

    def forward(self, x):

        training = self.model.training

        self.model.eval()

        out = self.model(x)

        if training:
            self.model.train()

        return out

In [17]:
model = TomatoGoogLeNet(num_classes=10).to(device)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torchvision/models/googlenet.py:341: UserWarning: auxiliary heads in the pretrained googlenet model are NOT pretrained, so make sure to train them
  warnings.warn(


In [18]:
total_params = sum(p.numel() for p in model.parameters())

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Total Parameters     : {total_params:,}")
print(f"Trainable Parameters : {trainable_params:,}")

Total Parameters     : 11,990,138
Trainable Parameters : 11,990,138


In [19]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.1,
    patience=3
)

epochs = 20

best_val_loss = float("inf")

patience = 5

counter = 0

In [20]:
from tqdm import tqdm
import torch

epochs = 20

best_val_loss = float("inf")
patience = 5
counter = 0

for epoch in range(epochs):

    
    # Training
    
    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in tqdm(train_loader,
                               desc=f"Epoch {epoch+1}/{epochs}"):

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_loss /= len(train_loader)
    train_acc = 100 * train_correct / train_total

    
    # Validation
   
    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_loss /= len(val_loader)
    val_acc = 100 * val_correct / val_total

    
    # Scheduler
    
    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch [{epoch+1}/{epochs}] | "
        f"LR: {current_lr:.6f} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.2f}%"
    )

   
    # Save Best Model
   
    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(
            model.state_dict(),
            "best_googlenet.pth"
        )

        counter = 0

        print("Best model saved.")

    else:

        counter += 1

        print(f"Early Stopping Counter: {counter}/{patience}")

        if counter >= patience:

            print("Early stopping triggered.")

            break

print("Training Complete.")

Epoch 1/20:   0%|                                       | 0/455 [00:00<?, ?it/s]/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Epoch 1/20:  82%|███████████████████████▊     | 374/455 [01:45<00:22,  3.55it/s]


KeyboardInterrupt: 

In [21]:
from sklearn.metrics import (

    confusion_matrix,

    classification_report,

    accuracy_score,

    precision_score,

    recall_score,

    f1_score

)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

model = TomatoGoogLeNet(num_classes=10).to(device)

model.load_state_dict(

    torch.load(

        "best_googlenet.pth",

        map_location=device

    )

)

model.eval()

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torchvision/models/googlenet.py:341: UserWarning: auxiliary heads in the pretrained googlenet model are NOT pretrained, so make sure to train them
  warnings.warn(


RuntimeError: Error(s) in loading state_dict for TomatoGoogLeNet:
	Missing key(s) in state_dict: "model.conv1.conv.weight", "model.conv1.bn.weight", "model.conv1.bn.bias", "model.conv1.bn.running_mean", "model.conv1.bn.running_var", "model.conv2.conv.weight", "model.conv2.bn.weight", "model.conv2.bn.bias", "model.conv2.bn.running_mean", "model.conv2.bn.running_var", "model.conv3.conv.weight", "model.conv3.bn.weight", "model.conv3.bn.bias", "model.conv3.bn.running_mean", "model.conv3.bn.running_var", "model.inception3a.branch1.conv.weight", "model.inception3a.branch1.bn.weight", "model.inception3a.branch1.bn.bias", "model.inception3a.branch1.bn.running_mean", "model.inception3a.branch1.bn.running_var", "model.inception3a.branch2.0.conv.weight", "model.inception3a.branch2.0.bn.weight", "model.inception3a.branch2.0.bn.bias", "model.inception3a.branch2.0.bn.running_mean", "model.inception3a.branch2.0.bn.running_var", "model.inception3a.branch2.1.conv.weight", "model.inception3a.branch2.1.b

In [22]:
all_labels = []
all_predictions = []

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predicted.cpu().numpy())

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


In [23]:
accuracy = accuracy_score(all_labels, all_predictions)

precision = precision_score(
    all_labels,
    all_predictions,
    average="weighted"
)

recall = recall_score(
    all_labels,
    all_predictions,
    average="weighted"
)

f1 = f1_score(
    all_labels,
    all_predictions,
    average="weighted"
)

print(f"Accuracy : {accuracy*100:.2f}%")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

Accuracy : 8.90%
Precision: 0.1663
Recall   : 0.0890
F1 Score : 0.1004


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [24]:
report = classification_report(
    all_labels,
    all_predictions,
    target_names=train_data.classes,
    digits=4
)

print(report)

with open("classification_report.txt", "w") as f:
    f.write(report)

                                               precision    recall  f1-score   support

                      Tomato___Bacterial_spot     0.0323    0.0047    0.0082       425
                        Tomato___Early_blight     0.0000    0.0000    0.0000       200
                         Tomato___Late_blight     0.1667    0.0026    0.0052       382
                           Tomato___Leaf_Mold     0.0556    0.0105    0.0176       191
                  Tomato___Septoria_leaf_spot     0.1795    0.0198    0.0356       354
Tomato___Spider_mites Two-spotted_spider_mite     0.0000    0.0000    0.0000       335
                         Tomato___Target_Spot     0.0000    0.0000    0.0000       281
       Tomato___Tomato_Yellow_Leaf_Curl_Virus     0.4215    0.2558    0.3184      1071
                 Tomato___Tomato_mosaic_virus     0.0138    0.5000    0.0268        74
                             Tomato___healthy     0.0000    0.0000    0.0000       318

                                     accu

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this beha

In [25]:
cm = confusion_matrix(
    all_labels,
    all_predictions
)

plt.figure(figsize=(10,8))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=train_data.classes,
    yticklabels=train_data.classes
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("GoogLeNet Confusion Matrix")

plt.tight_layout()

plt.savefig(
    "googlenet_confusion_matrix.png",
    dpi=300
)

plt.show()

NameError: name 'plt' is not defined

In [ ]:
per_class_accuracy = cm.diagonal() / cm.sum(axis=1)

df = pd.DataFrame({

    "Class": train_data.classes,

    "Accuracy (%)": per_class_accuracy * 100

})

print(df)

df.to_csv(
    "per_class_accuracy.csv",
    index=False
)

In [26]:
import torchvision.transforms.v2 as T

tta_transforms = [

    T.Compose([
        T.Resize((224,224)),
        T.ToImage(),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(mean.tolist(), std.tolist())
    ]),

    T.Compose([
        T.Resize((224,224)),
        T.RandomHorizontalFlip(p=1.0),
        T.ToImage(),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(mean.tolist(), std.tolist())
    ]),

    T.Compose([
        T.Resize((224,224)),
        T.RandomRotation(10),
        T.ToImage(),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(mean.tolist(), std.tolist())
    ])
]

In [ ]:
from PIL import Image
def tta_predict(image_path):

    image = Image.open(image_path).convert("RGB")

    probs = []

    for transform in tta_transforms:

        img = transform(image)

        img = img.unsqueeze(0).to(device)

        with torch.no_grad():

            output = model(img)

            probs.append(
                torch.softmax(output, dim=1)
            )

    probs = torch.stack(probs)

    return probs.mean(0)
    

In [ ]:
all_labels = []
all_predictions = []

samples = val_data.samples

for image_path, label in samples:

    prediction = tta_predict(image_path)

    pred = prediction.argmax(dim=1).item()

    all_predictions.append(pred)

    all_labels.append(label)
    

In [ ]:
tta_accuracy = accuracy_score(
    all_labels,
    all_predictions
)

print(
    f"TTA Accuracy : {tta_accuracy*100:.2f}%"
)